In [1]:
import csv
import io
import os
import requests

BASE_URL = "https://storage.googleapis.com/sefaria-export/links/"
OUTPUT_PREFIX = "commentary_talmud_links"
MAX_FILE_SIZE = 20 * 1024 * 1024  # 20 MB


def row_matches(row):
    if len(row) < 7:
        return False

    col3 = row[2].strip().lower()
    col6 = row[5].strip().lower()
    col7 = row[6].strip().lower()

    return col3 == "commentary" and ("talmud" in col6 or "talmud" in col7)


def file_missing(response):
    if response.status_code == 404:
        return True

    content_type = response.headers.get("Content-Type", "").lower()
    if "xml" in content_type:
        text = response.text
        if "<Code>NoSuchKey</Code>" in text or "The specified key does not exist" in text:
            return True

    return False


class RotatingCSVWriter:
    def __init__(self, prefix, max_size):
        self.prefix = prefix
        self.max_size = max_size
        self.file_index = 0
        self.file = None
        self.writer = None
        self.current_filename = None
        self.header = ["Citation 1", "Citation 2"]
        self._open_new_file()

    def _open_new_file(self):
        if self.file is not None:
            self.file.close()

        self.file_index += 1
        self.current_filename = f"{self.prefix}_{self.file_index}.csv"
        self.file = open(self.current_filename, "w", newline="", encoding="utf-8")
        self.writer = csv.writer(self.file)
        self.writer.writerow(self.header)
        self.file.flush()

    def _encoded_row_size(self, row):
        buf = io.StringIO()
        temp_writer = csv.writer(buf)
        temp_writer.writerow(row[:2])
        return len(buf.getvalue().encode("utf-8"))

    def writerow(self, row):
        output_row = row[:2]
        current_size = os.path.getsize(self.current_filename)
        next_row_size = self._encoded_row_size(output_row)

        if current_size + next_row_size > self.max_size and current_size > 0:
            self._open_new_file()

        self.writer.writerow(output_row)
        self.file.flush()

    def close(self):
        if self.file is not None:
            self.file.close()
            self.file = None


def process_file(url, session, out_writer):
    response = session.get(url, timeout=60)

    if file_missing(response):
        return False, 0, 0

    response.raise_for_status()

    lines = response.text.splitlines()
    reader = csv.reader(lines)
    next(reader, None)  # skip header row

    total_in = 0
    total_out = 0

    for row in reader:
        total_in += 1
        if row_matches(row):
            out_writer.writerow(row)
            total_out += 1

    return True, total_in, total_out


def main():
    session = requests.Session()
    out_writer = RotatingCSVWriter(OUTPUT_PREFIX, MAX_FILE_SIZE)

    file_num = 0
    grand_total_in = 0
    grand_total_out = 0

    try:
        while True:
            url = f"{BASE_URL}links{file_num}.csv"
            print(f"Checking {url}")

            exists, total_in, total_out = process_file(url, session, out_writer)
            if not exists:
                print(f"Stopping at links{file_num}.csv: file not found")
                break

            print(f"Processed links{file_num}.csv: read {total_in}, kept {total_out}")
            grand_total_in += total_in
            grand_total_out += total_out
            file_num += 1
    finally:
        out_writer.close()

    print(f"Done. Read {grand_total_in} rows, kept {grand_total_out} rows.")
    print(f"Created {out_writer.file_index} output file(s).")

In [2]:
main()

Checking https://storage.googleapis.com/sefaria-export/links/links0.csv
Processed links0.csv: read 298534, kept 28411
Checking https://storage.googleapis.com/sefaria-export/links/links1.csv
Processed links1.csv: read 299992, kept 75861
Checking https://storage.googleapis.com/sefaria-export/links/links2.csv
Processed links2.csv: read 299984, kept 44769
Checking https://storage.googleapis.com/sefaria-export/links/links3.csv
Processed links3.csv: read 299996, kept 79708
Checking https://storage.googleapis.com/sefaria-export/links/links4.csv
Processed links4.csv: read 299988, kept 14568
Checking https://storage.googleapis.com/sefaria-export/links/links5.csv
Processed links5.csv: read 299995, kept 5214
Checking https://storage.googleapis.com/sefaria-export/links/links6.csv
Processed links6.csv: read 299990, kept 40800
Checking https://storage.googleapis.com/sefaria-export/links/links7.csv
Processed links7.csv: read 299988, kept 2516
Checking https://storage.googleapis.com/sefaria-export/lin